# Projet Python pour la Data Science : Influence des médailles remportées par la France aux Jeux Olympiques sur le nombre le licenciés sportifs en France.

_Autrices : Melissa MIGAN, Camille PEYTHIEUX-TALDIR, Romane PLUQUET_.

## Introduction

# to do

### Sommaire

#TODO
[Introduction](#introduction)

[Données](#données)

[Analyse](#analyse)

[Conclusion](#conclusion)

## I. Création et exploration de la base de données principale

Dans cette partie, l'objectif est d'importer et de travailler les différentes bases de données et de les joindre en une base de données exploitable. Après travail et nettoyage des données brutes (en accès public), nous utilisons trois bases de données :

| Nom de la base   | Description                                                                 | Source     | Mode d'extraction |
|-----------------|------------------------------------------------------------------------------|------------|-------------------|
| `data_medailles`  | Nombre de médailles reportées par la France aux Jeux Olympiques par sport et année (2016-2024). | Wikipedia  | Web scraping      |
| `data_licences`   | Nombre de licenciés sportifs en France par fédération, sexe, âge et département (2016-2024).       | Injep (Institut national de la jeunesse et de l'éducation populaire)          | CSV, Parquet      |
| `data_pop`        | Population départementale en France (recensements de 2016 et 2022).          | Insee      | API               |

Ces tables regroupent les données suivantes :
- La table `data_medailles` rassemble les médailles obtenues par la France aux Jeux Olympiques entre 2016 et 2024, c'est-à-dire aux JO de 2016, 2020 (qui ont eu lieu en 2021 à cause du Covid) et de 2024. 
- La table `data_licences` recense les effectif de licenciés par an entre 2016 et 2024. Elle présente également des effectifs par tranches d'âge et par genre.
- La table `data_pop` contient les populations départementales recensées en 2016 et en 2022. Elle nous permet de mener une étude des effectifs de licenciés par département, relativement à la population de ces derniers. Cette table n'étant utilisée que dans un unique graphique (dans un but de pondération), elle ne sera pas incluse dans notre base de données principale.

On importe les modules pour traiter les données, et les fonctions utilisées.

In [1]:
#%pip install -r requirements.txt

# Modules
#import os
import pandas as pd
import numpy as np
import pyarrow as pa

# Fonctions
from data import (
    gel_tableau_medailles,
    nettoyer_base,
    fusionner_bases,
    reorganiser_colonnes,
    normalisation_unicode,
    code_sport,
    code_dep,
    renommer_colonnes, 
    gel_licences,
    tableau_ratios_nr
)

### A. Récupération des données

#### 1. Médailles françaises aux Jeux Olympiques

Nous avons scrappé la page Wikipedia ["_France aux Jeux Olympiques_"](https://fr.wikipedia.org/wiki/France_aux_Jeux_olympiques) afin d'obtenir les tableaux des médailles (or, argent bronze) obtenues par la France lors des Jeux Olympiques, à l'aide de la fonction `tableau_scraper`.

In [2]:
a_figer = [["or", "M.C3.A9dailles_d.27or_3"],
           ["argent", "M.C3.A9dailles_d.27argent"],
           ["bronze", "M.C3.A9dailles_de_bronze"]]

for duo in a_figer:
    #gel_tableau_medailles(duo[0], duo[1])

_IncompleteInputError: incomplete input (3881572216.py, line 6)

 Nous avons ensuite gelé les bases scrappées dans un souci de reproductibilité, au cas où la page Wikipédia soit modifiée. Nous avons ainsi obtenu trois tables :
- `data_or` : renseignant le nombre de médailles d'or obtenues par la France aux Jeux Olympiques, par sport et par année,
- `data_argent` : renseignant le nombre de médailles d'argent obtenues par la France aux Jeux Olympiques, par sport et par année,
- `data_bronze` : renseignant le nombre de médailles de bronze obtenues par la France aux Jeux Olympiques, par sport et par année.

In [3]:
data_or = pd.read_csv(f"data/data_medailles/data_or_jo.csv")
data_argent = pd.read_csv(f"data/data_medailles/data_argent_jo.csv")
data_bronze = pd.read_csv(f"data/data_medailles/data_bronze_jo.csv")

display(data_or.head())
display(data_argent.head())
display(data_bronze.head())

,Place,Unnamed: 1,Sport,1896-2024,2024,2020,2016,2012,2008,2004,...,1936,1932,1928,1924,1920,1912,1908,1904,1900,1896
0,1,NaN,Escrime,45.0,1.0,2.0,1.0,0.0,2.0,3.0,...,0.0,2.0,2.0,3.0,1.0,0.0,2.0,0.0,5.0,1.0
1,2,NaN,Cyclisme,44.0,3.0,0.0,0.0,1.0,2.0,1.0,...,3.0,1.0,1.0,4.0,1.0,0.0,1.0,0.0,2.0,4.0
2,3,NaN,Judo,18.0,2.0,2.0,2.0,2.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,NaN,Équitation,14.0,0.0,0.0,2.0,0.0,0.0,1.0,...,0.0,2.0,0.0,0.0,0.0,1.0,NaN,NaN,1.0,NaN
4,4,NaN,Athlétisme,14.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


,Place,Unnamed: 1,Sport,1896-2024,2024,2020,2016,2012,2008,2004,...,1936,1932,1928,1924,1920,1912,1908,1904,1900,1896
0,1,NaN,Escrime,47.0,4.0,2.0,1.0,0.0,2.0,1.0,...,2.0,1.0,3.0,3.0,4.0,0.0,1.0,0.0,5.0,2.0
1,2,NaN,Cyclisme,30.0,3.0,0.0,0.0,3.0,3.0,1.0,...,2.0,2.0,0.0,0.0,0.0,0.0,2.0,0.0,2.0,1.0
2,2,NaN,Athlétisme,28.0,1.0,1.0,3.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0.0,2.0,2.0,1.0,1.0,4.0,1.0
3,4,NaN,Natation,17.0,1.0,1.0,2.0,2.0,2.0,2.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
4,5,NaN,Aviron,15.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,3.0,1.0,0.0,0.0,0.0,3.0,NaN


,Place,Unnamed: 1,Sport,1896-2024,2024,2020,2016,2012,2008,2004,...,1936,1932,1928,1924,1920,1912,1908,1904,1900,1896
0,1,NaN,Escrime,38.0,2.0,1.0,1.0,0.0,0.0,2.0,...,1.0,0.0,0.0,0.0,3.0,0.0,1.0,0.0,5.0,0.0
1,2,NaN,Judo,34.0,6.0,3.0,1.0,5.0,2.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,NaN,Athlétisme,30.0,0.0,0.0,3.0,1.0,2.0,2.0,...,0.0,1.0,1.0,3.0,1.0,0.0,1.0,0.0,2.0,1.0
3,4,NaN,Cyclisme,28.0,3.0,2.0,1.0,0.0,1.0,1.0,...,2.0,1.0,0.0,2.0,1.0,0.0,2.0,0.0,1.0,1.0
4,5,NaN,Natation,22.0,2.0,0.0,1.0,1.0,3.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0


Nous avons ensuite nettoyé ces tables, en enlevant les années et sports ne nous intéressant pas, ainsi que les lignes et colonnes vides ou de total.

In [4]:
data_or_clean = nettoyer_base(data_or)
data_argent_clean = nettoyer_base(data_argent)
data_bronze_clean = nettoyer_base(data_bronze)

display(data_or_clean.head())
display(data_argent_clean.head())
display(data_bronze_clean.head())

,Sport,2024,2020,2016
0,Escrime,1.0,2.0,1.0
1,Cyclisme,3.0,0.0,0.0
2,Judo,2.0,2.0,2.0
3,Équitation,0.0,0.0,2.0
4,Athlétisme,0.0,0.0,0.0


,Sport,2024,2020,2016
0,Escrime,4.0,2.0,1.0
1,Cyclisme,3.0,0.0,0.0
2,Athlétisme,1.0,1.0,3.0
3,Natation,1.0,1.0,2.0
4,Aviron,0.0,1.0,0.0


,Sport,2024,2020,2016
0,Escrime,2.0,1.0,1.0
1,Judo,6.0,3.0,1.0
2,Athlétisme,0.0,0.0,3.0
3,Cyclisme,3.0,2.0,1.0
4,Natation,2.0,0.0,1.0


Suite à cela, nous avons joint ces trois tables afin d'obtenir la table `data_medailles_jo`, renseignant le nombre de médailles (toutes couleurs confondues) obtenues par la France aux Jeux Olympiques de 2016, 2020 et 2024.

Nous y avons ajouté trois variables :
- `code_sport`, un code permettant plus tard la jointure avec la table des licenciés sportifs en France,
- `total_medailles_2020`, une somme de toutes les médailles remportées lors des Jeux Olympiques de 2020,
- `total_medailles_2024`, une somme de toutes les médailles remportées lors des Jeux Olympiques de 2024.

In [5]:
data_medailles = fusionner_bases(data_or, data_argent, data_bronze).head()
display(data_medailles.head())

,code_sport,sport,2024_or,2020_or,2016_or,2024_argent,2020_argent,2016_argent,2024_bronze,2020_bronze,2016_bronze,total_medailles_2016,total_medailles_2020,total_medailles_2024
0,ATH,Athlétisme,0.0,0.0,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
1,AVI,Aviron,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,2.0,2.0,0.0
2,BAD,Badminton,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,BAK,Basket-ball,0.0,0.0,0.0,3.0,1.0,0.0,0.0,1.0,0.0,0.0,2.0,3.0
4,BOX,Boxe,0.0,0.0,2.0,2.0,0.0,2.0,1.0,0.0,2.0,6.0,0.0,3.0


#### 2. Licenciés sportifs en France

Nous avons exploité les données de licenciés en France fournies par l'Injep, l'Institut national de la Jeunesse et de l'Education populaire. Les données se présentent sous forme brute comme un fichier CSV par an. Cependant, ces fichiers étant trop lourds pour être importés directement dans Git, nous optons pour une gestion des données par fichiers au format parquet. 


##### a. Construction de la base 

Notre but ici est de contruire une base de données au format long. Chaque base disposant déjà d'une colonne année, nous devons alors les concaténer pour obtenir la base au format désiré. Pour que l'opération se déroule correctement, nous réorganisons dans un premier temps les colonnes de chacune des bases, de sorte que chacune ait les mêmes colonnes dans le même ordre.

In [6]:
liste_fichiers = ["data/data_licences/Lics_2016_semidef.parquet",
                  "data/data_licences/Lics_2017_semidef.parquet", 
                  "data/data_licences/Lics_2018_semidef.parquet",
                  "data/data_licences/Lics_2019_def.parquet",
                  "data/data_licences/Lics_2020_def.parquet",
                  "data/data_licences/Lics_2021_def.parquet",
                  "data/data_licences/Lics_2022_def.parquet",
                  "data/data_licences/Lics_2023_semidef.parquet",
                  "data/data_licences/Lics_2024_semidef.parquet"]

data_licences = reorganiser_colonnes(liste_fichiers)
data_licences = pa.concat_tables(data_licences)

Afin d'éviter les problèmes de sélection de données, car nous disposons de variables dont les modalités sont textuelles, nous normalisons tous les caractères avec la norme unicode. Nous ajoutons ensuite plusieurs variables permettant un niveau d'analyse plus général que celui très fin proposé par la base de données : 
- `code_sport` : catégorise les fédérations selon le sport pratiqué. Nous choisissons de catégoriser en "divers" (`code_sport` : DIV) les fédérations qui pratiquent un sport non-olympique. Ce code est le même que celui ajouté à la table des médailles. 
- `code_dep` : indique le département par son numéro seulement. 

In [7]:
data_licences = normalisation_unicode(data_licences)
data_licences = code_sport(data_licences)
data_licences = code_dep(data_licences, "Département")

Nous renommons ensuite les colonnes pour avoir des noms de variable sans majusucles ni accents. Nous ne gardons que les colonnes qui nous seront utiles pour la suite, à savoir celles concernant le nom de la fédération, l'année de recensement des licences, le sexe des licencié.e.s, les tranches d'âge (age, tranches fines et grandes fines), le nombre de licences annuelles, le code sport et le code département. Finalement, nous gelons la table dans un fichier parquet pour la réutiliser par la suite telle que construite ici. 

In [8]:
#data_licences = renommer_colonnes(data_licences)
data_licences = data_licences[["federation","annee", "sexe", "age", "tranche_age","grande_tranche_age","licences_annuelles","code_sport","code_dep"]]
#gel_licences(data_licences)
display(data_licences.sample(5))
display(data_licences["federation"].describe())

,federation,annee,sexe,age,tranche_age,grande_tranche_age,licences_annuelles,code_sport,code_dep
7314980,Fédération Française Handisport,2024,H,46,j - de 45 à 49 ans,3 - Adultes (21-55),5,DIV,08
7342628,Union Nationale du Sport Scolaire (UNSS),2024,F,20,e - de 20 à 24 ans,2 - Jeunes (14-20),5,DIV,80
3970953,Fédération Sportive et Culturelle de France,2020,H,10,c - de 10 à 14 ans,1 - Enfants (1-13),49,DIV,69
3467336,Fédération Française de Ski,2020,H,71,o - de 70 à 74 ans,4 - Seniors (56-99),61,DIV,73
5131785,Fédération Française de Volley,2022,H,27,f - de 25 à 29 ans,3 - Adultes (21-55),4,VOL,40


Nous obtenons une base de données comprenant plus de 7,35 millions de lignes. 118 fédérations sportives y sont recensées, et ce sur neuf années, de 2016 à 2024. La variable `code_sport` nous permet de catégoriser facilement les fédérations selon le sport pratiqué. Nous choisissons la modalité "DIV" pour les sport non-olympiques. Ainsi, 33 sports olympiques sont présents dans notre base de données. 

Nous disposons de plusieurs variables permettant de distinguer les licenciés selon des critères socio-démographiques. 
- trois variables d'âge :  
    - la variable `age` qui donne l'âge précis des licenciés, 
    - la variable `tranche_age` qui répertorie les licenciés selon 18 tranches d'âges "fines", par exemple ceux ayant de 10 à 14 ans,
    - la variable `grande_tranche_age` qui classe les licenciés selon cinq tranches d'âge plus larges, comme par exemple la catégorie Adultes (21-55 ans),
- la variable `sexe`, qui présente deux modalités, H ou F, et permet de classer les licenciés selon leur genre,
- la variable `code_dep`, qui renseigne le département dans lequel sont enregistrés les licenciés.

Finalement, notre variable d'intérêt est la variable `licences_annuelles`, que nous pouvons agréger selon toutes les variables présentes dans la base, en prenant les précautions nécessaires, détaillées dans la partie suivante.

##### b. Précautions : comparabilité dans le temps

Nos données de licences proviennent de fichiers distincts pour chaque année recensée. Ces fichiers sont de deux types :
- `semidef` pour les années 2016 à 2018 et 2023 à 2024, qui n'ont pas encore été "géocodées",
- `def` pour les années 2019 à 2022, qui sont "géocodées".

La documentation de ces données affirme que les données sont comparables dans le temps à condition d'être agrégées par fédération. Ainsi, comme notre code sport est directement dérivé des fédérations, l'agrégation selon la variable `code_sport` ne posera pas de problème de compabilité. Cependant, il est précisé que les "données par sexe, et/ou par âge, et/ou par département/région ne doivent pas être comparées dans le temps directement". Il faut dans ce cas là bien prendre en compte les effectifs non répartis (NR), c'est-à-dire qui n'ont pas pu être classés selon un département, une catégorie d'âge ou de genre, afin d'obtenir des résultats comparables dans le temps. 

Pour d'avoir une idée de l'ampleur de la non répartition géographique des effectifs de licences dans les données, nous calculons les ratios d'effectifs de licences géographiquement non répartis sur la totalité des effectifs de licences par an, ainsi que dans la base regroupant toutes les années. 

In [9]:
display(tableau_ratios_nr(data_licences, "code_dep"))

,2016,2017,2018,2019,2020,2021,2022,2023,2024,Global
Ratio de non répartis code_dep,4.78 %,5.19 %,4.45 %,0.74 %,0.76 %,0.91 %,0.50 %,0.47 %,0.40 %,2.05 %


On constate alors que la proportion de licences non géographiquement réparties est bien plus importante pour les premiers fichiers `semidef`, c'est-à-dire de 2016 à 2018, alors que par la suite, cette proportion n'excède pas les 1%. Ainsi, dès que nous mènerons une analyse géographique, nous prendrons en compte dans l'élaboration et l'analyse des résultats le fait que, de 2016 à 2018, autour de 5% des effectifs de licences ne sont géographiquement pas attribués, dans la mesure où cette proportion est significativement importante. Nous montrons ensuite que, comme le suggère la précision dans la documentation sur le fait que certains fichiers soient "géocodés" ou non, la non répartition géographique est la plus importante dans notre jeu de données, grâce à un calcul de ratio similaire. 

In [10]:
display(tableau_ratios_nr(data_licences, "sexe"))
display(tableau_ratios_nr(data_licences, "age"))
display(tableau_ratios_nr(data_licences, "tranche_age"))
display(tableau_ratios_nr(data_licences, "grande_tranche_age"))

,2016,2017,2018,2019,2020,2021,2022,2023,2024,Global
Ratio de non répartis sexe,2.43 %,2.39 %,0.90 %,2.79 %,0.00 %,0.00 %,0.00 %,0.00 %,0.00 %,0.97 %


,2016,2017,2018,2019,2020,2021,2022,2023,2024,Global
Ratio de non répartis age,1.16 %,0.69 %,2.96 %,0.34 %,0.25 %,0.31 %,0.03 %,0.01 %,0.01 %,0.65 %


,2016,2017,2018,2019,2020,2021,2022,2023,2024,Global
Ratio de non répartis tranche_age,1.16 %,0.69 %,2.96 %,0.34 %,0.25 %,0.31 %,0.03 %,0.01 %,0.01 %,0.65 %


,2016,2017,2018,2019,2020,2021,2022,2023,2024,Global
Ratio de non répartis grande_tranche_age,1.16 %,0.69 %,2.96 %,0.34 %,0.25 %,0.31 %,0.03 %,0.01 %,0.01 %,0.65 %


En ce qui concerne l'âge et le sexe, on constate que la non répartition est globalement moins importante, mais toujours assez marquée de 2016 à 2018. La non répartition pour les variables d'âge en tranches est strictement égale à celle de la variable d'âge, puisque ces dernières découlent directement de la première. La non répartition en terme d'âge n'excède pas les 2% par an, et tend vers de très faibles valeurs à partir de 2019 (< 0,31%). La non répartition par sexe, elle, n'excède pas les 2,43% et est nulle de 2020 à 2024. Ainsi, bien que ce phénomène soit moins important pour l'âge et le sexe, nous le prendrons en compte dans nos analyses par âge et par sexe. 

#### 3. Population départementale en France

Dans l'optique de mener une analyse des effectifs de licenciés par départements, proportionnellement à leur population, nous récupérons les données de population départementale en France grâce à l'API Melodi de l'INSEE. Nous choisissons d'utiliser le jeu de données de la population de municipale et non de la population de référence, car la population municipale est comptabilisée tous les ans, tout comme nos données de licences. Ce choix permet une cohérence temporelle entre nos données.
La documentation de l'API fournit directement le code permettant d'extraire les données voulues, que nous reprenons pour l'extraction des données. L'URL d'extraction a été obtenu par visualisation de la base de données au niveau de précision désiré, c'est-à-dire au niveau départemental; et permet de sélectionner directement les années d'intérêt (2016 à 2023, car les données de 2024 ne sont pas encore disponibles). La base de données extraite comporte 800 lignes : la population pour les 96 départements de France métropolitaines et quatre départements d'Outre-mer (Guadeloupe, Martinique, Guyane, La Réunion), sur huit années.

In [ ]:
url_api = "https://api.insee.fr/melodi/data/DS_POPULATIONS_HISTORIQUES?TIME_PERIOD=2016&TIME_PERIOD=2017&TIME_PERIOD=2018&TIME_PERIOD=2019&TIME_PERIOD=2020&TIME_PERIOD=2021&TIME_PERIOD=2022&TIME_PERIOD=2023&GEO=DEP"
data_pop = melodi_extraction(url_api)

Nous ne gardons que les variables utilies pour notre analyse, c'est-à-dire celles correspondant au département, à l'année et à la population comptablisée dans le département. Nous renommons également les colonnes avec le même format que pour les bases présentées précédemment. Finalement, nous ajoutons le même code département que dans les autres bases, à partir de la variable de département. 

In [ ]:
data_pop = data_pop[["GEO", "TIME_PERIOD", "OBS_VALUE_NIVEAU"]]
data_pop.columns = ["departement", "annee", "population"]
data_pop = code_dep_pop(data_pop, "departement")

Comme les données n'existent pas encore pour l'année 2024, nous utiliserons celles de 2023 pour des analyses relatives à la population en 2024. Nous nettoyons ensuite cette base, en forçant les types des variables (années en integers, départements en string), en supprimant les départements non cartographiables et en les triant par ordre croissant. Nous gelons ensuite les données dans un fichier CSV. 

In [ ]:
data_pop_clean = clean_population(data_pop)
#gel_population(data_pop_clean)

### B. Jointure des tables

Nous avons joint les tables médailles et licences pour pouvoir étudier en détail l'effet de remporter des médailles aux Jeux Olympiques sur l'évolution du nombre de licenciés sportifs. Nous avons joint par la gauche en utilisant la clé `code_sport` pour ne pas démultiplier le nombre de lignes dans notre DataFrame : nous avons un unique code sport par ligne dans norte table médailles.

In [11]:
data_complet = pd.merge(data_licences, data_medailles, how='left', on="code_sport")
data_complet.sample(5)

,federation,annee,sexe,age,tranche_age,grande_tranche_age,licences_annuelles,code_sport,code_dep,sport,...,2016_or,2024_argent,2020_argent,2016_argent,2024_bronze,2020_bronze,2016_bronze,total_medailles_2016,total_medailles_2020,total_medailles_2024
4176637,Fédération Française de Football,2021,F,51,k - de 50 à 54 ans,3 - Adultes (21-55),20,FOO,35,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4339234,Fédération Française de Triathlon et Disciplin...,2021,H,54,k - de 50 à 54 ans,3 - Adultes (21-55),7,TRI,47,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4274941,Fédération Française de Taekwondo et Disciplin...,2021,H,59,l - de 55 à 59 ans,4 - Seniors (56-99),2,TAE,81,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1455217,Fédération Française d'Éducation Physique et d...,2017,F,60,m - de 60 à 64 ans,4 - Seniors (56-99),43,DIV,62,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024282,Fédération Française d'Aïkido et de Budo,2018,F,54,k - de 50 à 54 ans,3 - Adultes (21-55),4,DIV,74,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


to do

In [12]:
#data_complet.to_parquet("data_complet.parquet")


OUTPUT_DIR.mkdir(exist_ok=True)

musees.to_csv(OUTPUT_DIR / "musees.csv", index=False)
frequentation_annuelle.to_csv(OUTPUT_DIR / "frequentation_annuelle.csv", index=False)
freq_excel_long.to_csv(OUTPUT_DIR / "frequentation_excel_long.csv", index=False)
df_modele_clean.to_csv(OUTPUT_DIR / "df_modele_musees.csv", index=False)

print("Fichiers exportés dans :", OUTPUT_DIR.resolve())


NameError: name 'OUTPUT_DIR' is not defined

### C. Contrôles qualité et tests

Chargement + aperçu global

In [40]:
#df = pd.read_parquet("data/data_complet.parquet")
df=data_complet

print("Dimensions :", df.shape)
df.head()

Dimensions : (7354671, 22)


,federation,annee,sexe,age,tranche_age,grande_tranche_age,licences_annuelles,code_sport,code_dep,sport,...,2016_or,2024_argent,2020_argent,2016_argent,2024_bronze,2020_bronze,2016_bronze,total_medailles_2016,total_medailles_2020,total_medailles_2024
0,Fédération Française d'Athlétisme,2016,F,5,b - de 5 à 9 ans,1 - Enfants (1-13),1,ATH,01,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
1,Fédération Française d'Athlétisme,2016,F,6,b - de 5 à 9 ans,1 - Enfants (1-13),13,ATH,01,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
2,Fédération Française d'Athlétisme,2016,F,7,b - de 5 à 9 ans,1 - Enfants (1-13),28,ATH,01,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
3,Fédération Française d'Athlétisme,2016,F,8,b - de 5 à 9 ans,1 - Enfants (1-13),49,ATH,01,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
4,Fédération Française d'Athlétisme,2016,F,9,b - de 5 à 9 ans,1 - Enfants (1-13),76,ATH,01,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0


In [41]:
df.dtypes

federation               object
annee                     int64
sexe                     object
age                      object
tranche_age              object
grande_tranche_age       object
licences_annuelles        int64
code_sport               object
code_dep                 object
sport                    object
2024_or                 float64
2020_or                 float64
2016_or                 float64
2024_argent             float64
2020_argent             float64
2016_argent             float64
2024_bronze             float64
2020_bronze             float64
2016_bronze             float64
total_medailles_2016    float64
total_medailles_2020    float64
total_medailles_2024    float64
dtype: object

In [42]:
na_table = (
    df.isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("taux_manquants")
)

na_table


,taux_manquants
2020_or,0.926157
2016_or,0.926157
total_medailles_2020,0.926157
total_medailles_2016,0.926157
2016_bronze,0.926157
2020_bronze,0.926157
2024_bronze,0.926157
2016_argent,0.926157
2020_argent,0.926157
2024_argent,0.926157


NaN dans ce qui vient du df_medailles : normal c'est à cause du merge. NaN dans code département correspond aux fédérations sportives de l'étranger.

In [43]:
df[df["code_dep"].isna()]

,federation,annee,sexe,age,tranche_age,grande_tranche_age,licences_annuelles,code_sport,code_dep,sport,...,2016_or,2024_argent,2020_argent,2016_argent,2024_bronze,2020_bronze,2016_bronze,total_medailles_2016,total_medailles_2020,total_medailles_2024
7237,Fédération Française d'Athlétisme,2016,F,7,b - de 5 à 9 ans,1 - Enfants (1-13),1,ATH,NaN,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
7238,Fédération Française d'Athlétisme,2016,F,8,b - de 5 à 9 ans,1 - Enfants (1-13),1,ATH,NaN,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
7239,Fédération Française d'Athlétisme,2016,F,9,b - de 5 à 9 ans,1 - Enfants (1-13),1,ATH,NaN,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
7240,Fédération Française d'Athlétisme,2016,F,10,c - de 10 à 14 ans,1 - Enfants (1-13),3,ATH,NaN,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
7241,Fédération Française d'Athlétisme,2016,F,11,c - de 10 à 14 ans,1 - Enfants (1-13),1,ATH,NaN,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7344973,Union Nationale du Sport Scolaire (UNSS),2024,H,17,d - de 15 à 19 ans,2 - Jeunes (14-20),170,DIV,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7344974,Union Nationale du Sport Scolaire (UNSS),2024,H,18,d - de 15 à 19 ans,2 - Jeunes (14-20),16,DIV,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7344975,Union Nationale du Sport Scolaire (UNSS),2024,H,19,d - de 15 à 19 ans,2 - Jeunes (14-20),1,DIV,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7344976,Union Nationale du Sport Scolaire (UNSS),2024,H,33,g - de 30 à 34 ans,3 - Adultes (21-55),2,DIV,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
for col in ["sexe", "age"]:
    print(f"\n{col.upper()}")
    print(df[col].value_counts(normalize=True))



SEXE
sexe
H                   0.562956
F                   0.436777
NR - Non réparti    0.000266
Name: proportion, dtype: float64

AGE
age
15    0.014943
14    0.014909
16    0.014905
17    0.014722
13    0.014668
        ...   
95    0.000215
96    0.000137
97    0.000082
98    0.000054
99    0.000046
Name: proportion, Length: 100, dtype: float64


contrôle de domaine

années

In [45]:
df["annee"].min(), df["annee"].max()

(np.int64(2016), np.int64(2024))

nbr de licences

In [46]:
(df["licences_annuelles"] < 0).sum()

np.int64(0)

In [47]:
df["licences_annuelles"].describe()

count    7.354671e+06
mean     1.939609e+01
std      1.488199e+02
min      0.000000e+00
25%      1.000000e+00
50%      4.000000e+00
75%      1.100000e+01
max      1.518910e+05
Name: licences_annuelles, dtype: float64

depts

In [48]:
import re

regex_dep = re.compile(r"^(2A|2B|\d{2,3})$")

df["code_dep"].dropna().apply(
    lambda x: bool(regex_dep.match(x))
).value_counts()


code_dep
True    7295789
Name: count, dtype: int64

pas de false : tout va bien !

doublons ?

In [50]:
cles = ["annee", "federation", "licences_annuelles", "code_dep", "sport", "sexe", "age"]

nb_doublons = df.duplicated(subset=cles).sum()
print("Nombre de doublons :", nb_doublons)


Nombre de doublons : 2006


In [53]:
df.groupby(
    ["annee","code_dep","sport","sexe","age"]
)["licences_annuelles"].count().describe()


count    540087.000000
mean          1.000046
std           0.006803
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           2.000000
Name: licences_annuelles, dtype: float64

In [51]:
import geopandas as gpd

gdf_dep = gpd.read_file("departements.geojson")

dep_licences = set(df["code_dep"].dropna().unique())
dep_geo = set(gdf_dep["code"].unique())

taux_match = len(dep_licences & dep_geo) / len(dep_licences)
print(f"Taux de correspondance licences ↔ géométrie : {taux_match:.2%}")


Taux de correspondance licences ↔ géométrie : 88.89%


In [54]:
sorted(dep_licences - dep_geo)


['971',
 '972',
 '973',
 '974',
 '975',
 '976',
 '977',
 '978',
 '980',
 '986',
 '987',
 '988']

pk seulement 89??? ETR, outre mer

“Le taux de correspondance inférieur à 100% s’explique par la présence de modalités non territorialisables (NR, agrégats nationaux, territoires hors champ de la carte), qui sont volontairement conservées dans la base complète.”

In [52]:
n_avant = len(df)
df_cartes = df.dropna(subset=["code_dep"])
n_apres = len(df_cartes)

print(f"Lignes supprimées pour la cartographie : {n_avant - n_apres}")
print(f"Soit {(n_avant - n_apres)/n_avant:.2%} de la base")


Lignes supprimées pour la cartographie : 58882
Soit 0.80% de la base


Doublons apparents
La base n’est pas unique à la granularité (année, département, sport, sexe, âge), ce qui est attendu compte tenu du niveau de détail des données sources. Une agrégation à cette granularité est réalisée lorsque nécessaire (cartographie, modélisation).
Couverture géographique
Environ 89% des codes départementaux présents dans les données de licences correspondent à une entité géographique. Les écarts s’expliquent par la présence de modalités non territorialisables (NR, agrégats nationaux, territoires hors champ), conservées dans la base complète mais exclues des visualisations cartographiques.
Impact du nettoyage
L’exclusion des lignes sans code département valide concerne moins de 1% des observations, ce qui limite fortement le risque de biais dans les analyses spatiales.

## II. Visualisations et statistiques descriptives